# 🏋️ 운동 보조 시스템 v5 — 피드백 가독성 업그레이드
**캡스톤 디자인 | 스쿼트 기준 | 원본비율 유지 + 전문가 skeleton + 몸통 zone + 안정화 HUD**

---
### ✅ v5에서 달라진 점
1. **피드백 문장 개선**: `값이 전문가보다 큽니다/작습니다` 대신 `몸통 기울기 과도`, `무릎이 덜 굽혀짐`처럼 의미 중심 문장으로 표시합니다.
2. **불안정 지표 제외**: 측면영상에서 흔들리기 쉬운 `neck_lean`, `foot_width`, `hand_width` 계열은 핵심 오류판정에서 제외합니다.
3. **피드백 안정화**: 같은 오류가 3프레임 정도 연속될 때만 핵심 피드백을 바꾸어 깜빡임을 줄입니다.
4. **HUD 개선**: 현재 구간, `U=사용자 / E=전문가 / Δ=차이` 설명, 핵심 피드백 카드를 크게 표시합니다.
5. **빨간 원 최소화**: 현재 우선 확인해야 하는 관절만 표시해 화면이 덜 복잡하게 보이도록 했습니다.

---
### 실행 순서
Cell 1부터 Cell 10까지 순서대로 실행하면 됩니다. Cell 7.5가 v5 핵심 업그레이드 셀입니다.


In [ ]:
# ── Cell 2: 버전 확인 ─────────────────────────────────────────────
import numpy as np
import cv2
import mediapipe as mp
from PIL import Image

print('numpy    :', np.__version__)
print('cv2      :', cv2.__version__)
print('mediapipe:', mp.__version__)
print('Pillow   :', Image.__version__)

In [ ]:
# ── Cell 3: 경로 설정 + 촬영 방향 설정 ───────────────────────────
import os
import glob

# 네 구글 드라이브 기준 Capstone 폴더
BASE = '/content/drive/MyDrive/Capstone'

EXERCISE = 'squat'

# ✅ 촬영 방향 설정
# - squat_15.mp4처럼 완전 측면이면 'side'
# - 정면이면 'front'
# - 45도 사선이면 '45deg'
# - 자동 추정은 'auto' 가능하지만, 발표/실험용은 직접 지정이 가장 안전함.
VIDEO_VIEW_MODE = 'side'

# 입력 파일 경로
USER_VIDEO_PATH  = os.path.join(BASE, 'squat_15.mp4')
EXPERT_JSON_PATH = os.path.join(BASE, 'expert_squat.json')
MODEL_PATH       = os.path.join(BASE, 'pose_landmarker_lite.task')

# 출력 파일 경로
OUTPUT_DIR  = os.path.join(BASE, 'output')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'squat_v5_trunk_foot_upgrade.mp4')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 한글 폰트
FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

# 파일 존재 확인
for p, name in [
    (USER_VIDEO_PATH,  '사용자영상'),
    (EXPERT_JSON_PATH, '전문가JSON'),
    (MODEL_PATH,       'PoseLandmarker모델'),
]:
    ok = os.path.exists(p)
    print(f'{"OK" if ok else "없음"} | {name}: {p}')

if not os.path.exists(FONT_PATH):
    found = glob.glob('/usr/share/fonts/**/*Nanum*', recursive=True)
    FONT_PATH = found[0] if found else None
    print(f'폰트 재설정: {FONT_PATH}')
else:
    print(f'OK | 폰트: {FONT_PATH}')

print(f'촬영 방향 VIDEO_VIEW_MODE = {VIDEO_VIEW_MODE}')
print(f'출력 파일 = {OUTPUT_PATH}')


In [ ]:
# ── Cell 4: rt / fo import (Drive .py 파일) ──────────────────────
import sys
import importlib
import time
import math

if BASE not in sys.path:
    sys.path.insert(0, BASE)

import realtime_compare_side as rt
import feedback_overlay as fo

importlib.reload(rt)
importlib.reload(fo)

rt.EXERCISE         = EXERCISE
rt.EXPERT_JSON_PATH = EXPERT_JSON_PATH
rt.MODEL_PATH       = MODEL_PATH
rt.MIRROR_CAMERA    = False   # 저장 영상은 반전 없음

print('rt / fo import 완료')
print(f'  VISIBILITY_THRESHOLD = {rt.VISIBILITY_THRESHOLD}')
print(f'  WINDOW_SIZE          = {rt.WINDOW_SIZE}')
print(f'  EXPERT_ALPHA         = {rt.EXPERT_ALPHA}')

In [ ]:
# ── Cell 5: 공통지표 + fallback 기본지표 계산 함수 ───────────────
# 핵심 수정:
# 1) rt.compute_sideview_metrics 가 없어도 더 이상 터지지 않게 fallback 계산 추가
# 2) knee_angle / hip_angle / spine_lean 등 HUD에 필요한 기본 지표를 직접 계산
# 3) 사용자 landmark, 전문가 JSON landmark 모두 같은 함수로 처리

VIS_THR = getattr(rt, 'VISIBILITY_THRESHOLD', 0.4)


def get_xyv(lms, idx):
    """
    user lms(MediaPipe 객체 list) 또는 expert lms_raw(JSON list [[x,y,z,vis],...]) 양쪽 처리.
    반환: (np.array([x,y]), visibility)
    """
    p = lms[idx]
    if hasattr(p, 'x'):
        return (
            np.array([float(p.x), float(p.y)], dtype=np.float32),
            float(getattr(p, 'visibility', 1.0))
        )

    # JSON landmark: [x, y, z, visibility] 형태라고 가정
    x, y = float(p[0]), float(p[1])
    v    = float(p[3]) if len(p) > 3 else 1.0
    return np.array([x, y], dtype=np.float32), v


def midp(lms, a, b):
    """두 landmark 중점. 한쪽 visibility가 낮으면 더 잘 보이는 점 사용."""
    pa, va = get_xyv(lms, a)
    pb, vb = get_xyv(lms, b)

    if va >= VIS_THR and vb >= VIS_THR:
        return (pa + pb) / 2.0

    return pa if va >= vb else pb


def dist2d(a, b):
    return float(np.linalg.norm(a - b))


def angle_abc(a, b, c):
    """
    세 점 A-B-C가 만드는 각도.
    B가 꼭짓점.
    """
    ba = a - b
    bc = c - b
    n = np.linalg.norm(ba) * np.linalg.norm(bc)

    if n < 1e-9:
        return 0.0

    cosv = float(np.dot(ba, bc) / n)
    cosv = float(np.clip(cosv, -1.0, 1.0))
    return float(math.degrees(math.acos(cosv)))


def angle_to_vertical(a, b):
    """
    b -> a 벡터와 수직축 사이 각도.
    0도 = 수직에 가까움, 90도 = 수평에 가까움.
    """
    dx = float(a[0] - b[0])
    dy = float(a[1] - b[1])
    return abs(math.degrees(math.atan2(abs(dx), abs(dy) + 1e-9)))


def _vis_sum(lms, ids):
    total = 0.0
    for i in ids:
        try:
            _, v = get_xyv(lms, i)
            total += float(v)
        except Exception:
            pass
    return total


def _pick_visible_side(lms):
    """
    측면 영상에서는 한쪽 관절이 더 잘 잡히는 경우가 많으므로
    hip-knee-ankle visibility 합이 큰 쪽을 대표 side로 사용.
    """
    left_score  = _vis_sum(lms, [11, 13, 15, 23, 25, 27, 31])
    right_score = _vis_sum(lms, [12, 14, 16, 24, 26, 28, 32])

    if right_score > left_score:
        return {
            'shoulder': 12, 'elbow': 14, 'wrist': 16,
            'hip': 24, 'knee': 26, 'ankle': 28, 'foot': 32
        }

    return {
        'shoulder': 11, 'elbow': 13, 'wrist': 15,
        'hip': 23, 'knee': 25, 'ankle': 27, 'foot': 31
    }


def compute_base_sideview_metrics_local(lms, exercise='squat'):
    """
    realtime_compare_side.py 안의 compute_sideview_metrics가 없거나 실패할 때 쓰는 최소 fallback.
    HUD와 delta 계산에 필요한 핵심 지표를 직접 계산한다.
    """
    side = _pick_visible_side(lms)

    sh, _ = get_xyv(lms, side['shoulder'])
    el, _ = get_xyv(lms, side['elbow'])
    wr, _ = get_xyv(lms, side['wrist'])
    hp, _ = get_xyv(lms, side['hip'])
    kn, _ = get_xyv(lms, side['knee'])
    an, _ = get_xyv(lms, side['ankle'])
    ft, _ = get_xyv(lms, side['foot'])

    shoulder_c = midp(lms, 11, 12)
    hip_c      = midp(lms, 23, 24)
    torso_len  = max(dist2d(shoulder_c, hip_c), 1e-6)

    knee_angle  = angle_abc(hp, kn, an)
    hip_angle   = angle_abc(sh, hp, kn)
    ankle_angle = angle_abc(kn, an, ft)
    elbow_angle = angle_abc(sh, el, wr)

    spine_lean  = angle_to_vertical(shoulder_c, hip_c)
    trunk_lean  = spine_lean

    # hip center -> knee 벡터의 수직 기준 기울기. 기존 코드 호환용 proxy.
    hip_knee_angle = angle_to_vertical(kn, hip_c)

    # deadlift 계열 호환용 proxy
    hip_shoulder_y_ratio = abs(float(hip_c[1] - shoulder_c[1])) / torso_len

    # bench 계열 호환용 proxy
    wrist_elbow_x_ratio = abs(float(wr[0] - el[0])) / torso_len
    body_line_ratio = angle_to_vertical(shoulder_c, hip_c)

    return {
        'knee_angle':             float(knee_angle),
        'hip_angle':              float(hip_angle),
        'ankle_angle':            float(ankle_angle),
        'elbow_angle':            float(elbow_angle),
        'spine_lean':             float(spine_lean),
        'trunk_lean':             float(trunk_lean),
        'hip_knee_angle':         float(hip_knee_angle),
        'hip_shoulder_y_ratio':   float(hip_shoulder_y_ratio),
        'wrist_elbow_x_ratio':    float(wrist_elbow_x_ratio),
        'body_line_ratio':        float(body_line_ratio),
    }


def compute_common_metrics(lms, exercise='squat'):
    """
    공통지표 계산.
    모든 값은 2D normalized coordinate 기반 proxy.
    """
    nose,   _  = get_xyv(lms, 0)
    l_sh,   _  = get_xyv(lms, 11); r_sh,  _  = get_xyv(lms, 12)
    l_el,   _  = get_xyv(lms, 13); r_el,  _  = get_xyv(lms, 14)
    l_wr,   _  = get_xyv(lms, 15); r_wr,  _  = get_xyv(lms, 16)
    l_hip,  _  = get_xyv(lms, 23); r_hip, _  = get_xyv(lms, 24)
    l_kn,   _  = get_xyv(lms, 25); r_kn,  _  = get_xyv(lms, 26)
    l_an,   _  = get_xyv(lms, 27); r_an,  _  = get_xyv(lms, 28)
    l_heel, _  = get_xyv(lms, 29); r_heel,_  = get_xyv(lms, 30)
    l_foot, _  = get_xyv(lms, 31); r_foot,_  = get_xyv(lms, 32)

    shoulder_c = midp(lms, 11, 12)
    hip_c      = midp(lms, 23, 24)
    torso_len  = max(dist2d(shoulder_c, hip_c), 1e-6)

    left_foot_center  = (l_heel + l_foot) / 2.0
    right_foot_center = (r_heel + r_foot) / 2.0
    wrist_c           = (l_wr + r_wr) / 2.0

    foot_width             = dist2d(l_an, r_an) / torso_len
    foot_x_asymmetry       = abs(float(l_an[0] - r_an[0])) / torso_len
    foot_foreaft_asymmetry = abs(float(left_foot_center[0] - right_foot_center[0])) / torso_len

    left_foot_flatness     = abs(float(l_heel[1] - l_foot[1])) / torso_len
    right_foot_flatness    = abs(float(r_heel[1] - r_foot[1])) / torso_len
    foot_flatness          = max(left_foot_flatness, right_foot_flatness)

    hand_width           = dist2d(l_wr, r_wr) / torso_len
    hand_height_diff     = abs(float(l_wr[1] - r_wr[1])) / torso_len
    shoulder_height_diff = abs(float(l_sh[1] - r_sh[1])) / torso_len

    if exercise in ('squat', 'deadlift'):
        xs = [float(nose[0]), float(shoulder_c[0]), float(hip_c[0])]
        head_hip_line = (max(xs) - min(xs)) / torso_len
    else:
        ys = [float(nose[1]), float(shoulder_c[1]), float(hip_c[1])]
        head_hip_line = (max(ys) - min(ys)) / torso_len

    neck_lean             = angle_to_vertical(nose, shoulder_c)
    wrist_shoulder_y_diff = (float(wrist_c[1]) - float(shoulder_c[1])) / torso_len

    left_elbow_angle      = angle_abc(l_sh, l_el, l_wr)
    right_elbow_angle     = angle_abc(r_sh, r_el, r_wr)
    elbow_angle_avg       = (left_elbow_angle + right_elbow_angle) / 2.0

    trunk_lean = angle_to_vertical(shoulder_c, hip_c)

    return {
        'foot_width':              float(foot_width),
        'foot_x_asymmetry':        float(foot_x_asymmetry),
        'foot_foreaft_asymmetry':  float(foot_foreaft_asymmetry),
        'foot_flatness':           float(foot_flatness),
        'left_foot_flatness':      float(left_foot_flatness),
        'right_foot_flatness':     float(right_foot_flatness),
        'hand_width':              float(hand_width),
        'hand_height_diff':        float(hand_height_diff),
        'shoulder_height_diff':    float(shoulder_height_diff),
        'head_hip_line':           float(head_hip_line),
        'neck_lean':               float(neck_lean),
        'wrist_shoulder_y_diff':   float(wrist_shoulder_y_diff),
        'elbow_angle_avg':         float(elbow_angle_avg),
        'trunk_lean':              float(trunk_lean),
    }


def _call_rt_compute_sideview_metrics(lms, exercise='squat'):
    """
    rt.compute_sideview_metrics가 있으면 사용하되,
    없거나 signature가 달라도 notebook이 멈추지 않도록 처리.
    """
    fn = getattr(rt, 'compute_sideview_metrics', None)

    if not callable(fn):
        return {}

    for args in [(lms, exercise), (lms,)]:
        try:
            out = fn(*args)
            return dict(out) if isinstance(out, dict) else {}
        except TypeError:
            continue
        except Exception as e:
            print(f'[경고] rt.compute_sideview_metrics 실패 → fallback 사용: {e}')
            return {}

    return {}


def compute_user_metrics_plus(lms, exercise='squat'):
    """
    기존 rt 지표 + fallback 기본지표 + 공통지표 합산.
    rt.compute_sideview_metrics가 없어도 knee_angle 등이 반드시 나오게 만든다.
    """
    local_base = compute_base_sideview_metrics_local(lms, exercise)
    rt_base    = _call_rt_compute_sideview_metrics(lms, exercise)
    common     = compute_common_metrics(lms, exercise)

    merged = {}
    merged.update(local_base)
    merged.update(rt_base)   # rt에 같은 key가 있으면 rt 값을 우선 사용
    merged.update(common)    # 공통지표는 최종 반영
    return merged


def compute_expert_metrics_plus(ex_lms_raw, ex_metrics_json=None, exercise='squat'):
    """
    전문가 JSON 지표 + fallback 기본지표 + 공통지표 합산.
    expert JSON 내부 metrics가 비어 있어도 비교에 필요한 key를 만들어준다.
    """
    local_base = compute_base_sideview_metrics_local(ex_lms_raw, exercise)
    common     = compute_common_metrics(ex_lms_raw, exercise)

    json_base = {}
    if isinstance(ex_metrics_json, dict):
        json_base = dict(ex_metrics_json)

    merged = {}
    merged.update(local_base)
    merged.update(json_base)
    merged.update(common)
    return merged


print('공통지표 + fallback 기본지표 계산 함수 로드 완료')


In [ ]:
# ── Cell 6: 지표 설정 확장 ────────────────────────────────────────
# rt 모듈의 EXERCISE_METRICS / DELTA_THRESHOLDS / DELTA_TO_LM /
# DISPLAY_LABELS / METRIC_LABELS 를 공통지표 기반으로 확장.
# [실험적 기준] 표기 = heuristic threshold.

# ── compatibility patch: realtime_compare_side.py 구버전 보정 ──

if not hasattr(rt, 'DELTA_THRESHOLDS'):
    rt.DELTA_THRESHOLDS = {}

if not hasattr(rt, 'DELTA_TO_LM'):
    rt.DELTA_TO_LM = {}

if not hasattr(rt, 'DISPLAY_LABELS'):
    rt.DISPLAY_LABELS = {}

if not hasattr(rt, 'METRIC_LABELS'):
    rt.METRIC_LABELS = {}

if not hasattr(rt, 'EXERCISE_METRICS'):
    rt.EXERCISE_METRICS = {
        'squat': [],
        'deadlift': [],
        'bench': []
    }

print("구버전 realtime_compare_side.py 보정 완료")
ADDED_METRICS = {
    'squat': [
        'knee_angle', 'hip_angle', 'spine_lean', 'hip_knee_angle',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'elbow_angle_avg', 'trunk_lean',
    ],
    'deadlift': [
        'knee_angle', 'hip_angle', 'spine_lean', 'hip_shoulder_y_ratio',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'trunk_lean',
    ],
    'bench': [
        'elbow_angle', 'wrist_elbow_x_ratio', 'body_line_ratio',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'knee_angle',
    ],
}
rt.EXERCISE_METRICS.update(ADDED_METRICS)

# ✅ 중요: 측면영상에서는 정면/45도 전용 지표를 오류판정에서 제외한다.
# 발너비, 양손거리, 양손높이차, 어깨높이차는 측면영상에서 겹침/가림 때문에
# delta가 튀기 쉽다. 이 값들이 오류판정에 들어가면 기존 지표가 전부 깨져 보인다.
SIDE_RELIABLE_METRICS = {
    'squat': [
        'knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean',
        'hip_knee_angle', 'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
    'deadlift': [
        'knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean',
        'hip_shoulder_y_ratio', 'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
    'bench': [
        'elbow_angle', 'wrist_elbow_x_ratio', 'body_line_ratio',
        'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
}

FRONT_45_EXTRA_METRICS = {
    'squat': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
    'deadlift': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
    'bench': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
}

_view = str(globals().get('VIDEO_VIEW_MODE', 'side')).lower()
if _view in ['side', '측면']:
    rt.EXERCISE_METRICS[EXERCISE] = SIDE_RELIABLE_METRICS.get(EXERCISE, SIDE_RELIABLE_METRICS['squat'])
else:
    base = SIDE_RELIABLE_METRICS.get(EXERCISE, SIDE_RELIABLE_METRICS['squat'])
    extra = FRONT_45_EXTRA_METRICS.get(EXERCISE, [])
    rt.EXERCISE_METRICS[EXERCISE] = list(dict.fromkeys(base + extra))

# Delta threshold [실험적 기준] — 전문가와의 차이가 이 값을 넘으면 주의
rt.DELTA_THRESHOLDS.update({
    'foot_width':              0.15,
    'foot_x_asymmetry':        0.10,
    'foot_foreaft_asymmetry':  0.10,
    'foot_flatness':           0.04,
    'left_foot_flatness':      0.04,
    'right_foot_flatness':     0.04,
    'hand_width':              0.20,
    'hand_height_diff':        0.07,
    'shoulder_height_diff':    0.05,
    'head_hip_line':           0.10,
    'neck_lean':               10.0,
    'wrist_shoulder_y_diff':   0.15,
    'elbow_angle_avg':         12.0,
    'trunk_lean':              8.0,
})

# 관련 landmark 인덱스 (issue_marker 강조에 사용)
rt.DELTA_TO_LM.update({
    'foot_width':              [27, 28],
    'foot_x_asymmetry':        [27, 28],
    'foot_foreaft_asymmetry':  [27, 28, 29, 30, 31, 32],
    'foot_flatness':           [29, 30, 31, 32],
    'left_foot_flatness':      [29, 31],
    'right_foot_flatness':     [30, 32],
    'hand_width':              [15, 16],
    'hand_height_diff':        [15, 16],
    'shoulder_height_diff':    [11, 12],
    'head_hip_line':           [0, 11, 12, 23, 24],
    'neck_lean':               [0, 11, 12],
    'wrist_shoulder_y_diff':   [11, 12, 15, 16],
    'elbow_angle_avg':         [11, 12, 13, 14, 15, 16],
    'trunk_lean':              [11, 12, 23, 24],
})

DISPLAY_LABELS_KR = {
    'foot_width':              '발너비',
    'foot_x_asymmetry':        '발목X차',
    'foot_foreaft_asymmetry':  '발전후차',
    'foot_flatness':           '발바닥밀착',
    'left_foot_flatness':      '왼발밀착',
    'right_foot_flatness':     '오른발밀착',
    'hand_width':              '양손거리',
    'hand_height_diff':        '양손높이차',
    'shoulder_height_diff':    '어깨높이차',
    'head_hip_line':           '머리-골반정렬',
    'neck_lean':               '목머리각',
    'wrist_shoulder_y_diff':   '손목어깨높이',
    'elbow_angle_avg':         '팔꿈치각',
    'trunk_lean':              '몸통기울기',
    'knee_angle':              '무릎각',
    'hip_angle':               '엉덩이각',
    'spine_lean':              '척추기울기',
}
rt.DISPLAY_LABELS.update(DISPLAY_LABELS_KR)
rt.METRIC_LABELS.update(rt.DISPLAY_LABELS)

print('지표 설정 확장 완료')
print('현재 오류판정 사용 지표:', rt.EXERCISE_METRICS.get(EXERCISE, []))


In [ ]:
# ── Cell 7: 시각화 헬퍼 v4 ───────────────────────────────────────
# 목표:
# 1) 몸통 기울기 텍스트를 사람 몸 위에 작게 쓰지 않고, HUD에서 크게 설명
# 2) 한글 폰트가 없으면 네모(□□□)로 깨지지 않도록 영어 fallback 사용
# 3) 측면영상에서는 발너비를 오류판정/표시에서 제외
# 4) 기존 레이어와 새 레이어가 겹쳐서 지저분해지는 문제 완화

from PIL import Image as PilImage, ImageDraw, ImageFont
import glob
import os
import textwrap

# Skeleton 연결 정의
SKEL_CONNS = [
    (0,11),(0,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(24,26),(26,28),
    (27,29),(29,31),(28,30),(30,32),
]

# 색상 팔레트 (BGR)
C_EXP_BONE  = (80, 220, 120)
C_EXP_JOINT = (180, 255, 180)
C_BRACKET   = (0, 215, 255)
C_TRUNK_OK  = (80, 220, 100)
C_TRUNK_ERR = (60, 60, 255)
C_TRUNK_REF = (170, 170, 170)
C_ERR_CIR   = (0, 0, 255)

# ✅ 몸통 기울기 허용 오차
TRUNK_LEAN_TOL_DEG = 8.0

# 발너비 브라켓 표시 대상 뷰
FOOT_BRACKET_VIEW_MODES = {'front', '45deg', '45', 'diagonal', 'oblique', '정면', '45도', '사선'}


# ────────────────────────────────────────────────────────────────
# [한글 폰트 탐색]
# ────────────────────────────────────────────────────────────────
def find_korean_font():
    candidates = []

    # Cell 3에서 FONT_PATH를 지정했다면 먼저 확인
    try:
        if FONT_PATH:
            candidates.append(FONT_PATH)
    except NameError:
        pass

    # Colab / Ubuntu에서 자주 쓰는 경로들
    candidates += [
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
        '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]

    # 혹시 경로가 달라도 찾기
    patterns = [
        '/usr/share/fonts/**/*Nanum*.ttf',
        '/usr/share/fonts/**/*NotoSansCJK*.ttc',
        '/content/**/*Nanum*.ttf',
        '/content/**/*NotoSansCJK*.ttc',
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat, recursive=True))

    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None


KOREAN_FONT_PATH = find_korean_font()
FONT_KR_OK = KOREAN_FONT_PATH is not None

def _fnt(size):
    try:
        if FONT_KR_OK:
            return ImageFont.truetype(KOREAN_FONT_PATH, size)
    except Exception:
        pass
    return ImageFont.load_default()

def _ko(kr, en):
    """한글 폰트가 없으면 영어로 표시해서 네모 깨짐을 방지."""
    return kr if FONT_KR_OK else en


def draw_text_box_bgr(frame, lines, xy=(16, 16), font_size=20,
                      fg=(255,255,255), bg=(20,20,30), pad=8,
                      alpha=0.72):
    """
    BGR frame 위에 PIL로 읽기 쉬운 박스형 텍스트를 그림.
    lines: str 또는 list[str]
    fg/bg: RGB 기준
    """
    if isinstance(lines, str):
        lines = [lines]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    base = PilImage.fromarray(rgb).convert('RGBA')
    overlay = PilImage.new('RGBA', base.size, (0,0,0,0))
    drw = ImageDraw.Draw(overlay)

    font = _fnt(font_size)
    line_h = font_size + 6
    widths = []
    for line in lines:
        try:
            bbox = drw.textbbox((0,0), line, font=font)
            widths.append(bbox[2] - bbox[0])
        except Exception:
            widths.append(len(line) * font_size * 0.6)

    box_w = int(max(widths) + pad*2)
    box_h = int(len(lines)*line_h + pad*2)
    x, y = xy
    x = max(0, min(int(x), frame.shape[1] - box_w - 2))
    y = max(0, min(int(y), frame.shape[0] - box_h - 2))

    bg_rgba = (bg[0], bg[1], bg[2], int(255*alpha))
    try:
        drw.rounded_rectangle([x, y, x+box_w, y+box_h], radius=10, fill=bg_rgba)
    except Exception:
        drw.rectangle([x, y, x+box_w, y+box_h], fill=bg_rgba)

    for i, line in enumerate(lines):
        drw.text((x+pad, y+pad+i*line_h), line, font=font, fill=fg+(255,))

    composed = PilImage.alpha_composite(base, overlay).convert('RGB')
    return cv2.cvtColor(np.array(composed), cv2.COLOR_RGB2BGR)


# ────────────────────────────────────────────────────────────────
# [전문가 skeleton -> 별도 canvas]
# ────────────────────────────────────────────────────────────────
def draw_expert_on_canvas(canvas_w, canvas_h, ex_lms_raw, bad_indices=None):
    if bad_indices is None:
        bad_indices = set()

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    if ex_lms_raw is None or len(ex_lms_raw) < 33:
        cv2.putText(canvas, 'NO EXPERT DATA',
                    (canvas_w//2-80, canvas_h//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (80,80,80), 2)
        return canvas

    def to_px(idx):
        p = ex_lms_raw[idx]
        if hasattr(p, 'x'):
            x, y, v = float(p.x), float(p.y), float(getattr(p, 'visibility', 1.0))
        else:
            x, y = float(p[0]), float(p[1])
            v = float(p[3]) if len(p) > 3 else 1.0
        return (int(x * canvas_w), int(y * canvas_h)), v

    for (a, b) in SKEL_CONNS:
        pa, va = to_px(a)
        pb, vb = to_px(b)
        if va < VIS_THR or vb < VIS_THR:
            continue
        cv2.line(canvas, pa, pb, C_EXP_BONE, 2, cv2.LINE_AA)

    for idx in range(33):
        pt, v = to_px(idx)
        if v < VIS_THR:
            continue
        is_bad = idx in bad_indices
        r  = 6 if is_bad else 4
        c  = C_ERR_CIR if is_bad else C_EXP_JOINT
        cv2.circle(canvas, pt, r, c, -1, cv2.LINE_AA)
        if is_bad:
            cv2.circle(canvas, pt, r + 5, C_ERR_CIR, 2, cv2.LINE_AA)

    cv2.putText(canvas, 'EXPERT', (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, C_EXP_BONE, 2, cv2.LINE_AA)
    return canvas


# ── 발너비 bracket 표시 여부 판단 ────────────────────────────────
def infer_view_mode_from_pose(lms):
    """양발목 x거리/torso 비율로 정면/측면을 아주 거칠게 추정."""
    try:
        la, va = get_xyv(lms, 27)
        ra, vr = get_xyv(lms, 28)
        sc = midp(lms, 11, 12)
        hc = midp(lms, 23, 24)
        torso = max(dist2d(sc, hc), 1e-6)
        foot_x_gap = abs(float(la[0] - ra[0])) / torso
        if foot_x_gap > 0.45:
            return 'front'
        if foot_x_gap > 0.22:
            return '45deg'
        return 'side'
    except Exception:
        return 'side'


def should_draw_foot_bracket(lms=None, view_mode=None):
    vm = str(view_mode or globals().get('VIDEO_VIEW_MODE', 'side')).lower()
    if vm == 'auto':
        vm = infer_view_mode_from_pose(lms) if lms is not None else 'side'
    return vm in FOOT_BRACKET_VIEW_MODES


# ── 발너비 bracket 시각화 ────────────────────────────────────────
def draw_foot_bracket(frame, lms, metrics, view_mode=None):
    """
    정면/45도에서만 양발목 아래에 bracket 표시.
    측면영상에서는 발이 겹쳐 보이기 때문에 표시하지 않는다.
    """
    if not should_draw_foot_bracket(lms, view_mode=view_mode):
        return frame

    H, W = frame.shape[:2]
    la, va = get_xyv(lms, 27)
    ra, vr = get_xyv(lms, 28)
    if va < VIS_THR or vr < VIS_THR:
        return frame

    lx = int(la[0] * W);  ly = int(la[1] * H)
    rx = int(ra[0] * W);  ry = int(ra[1] * H)
    y   = min(H - 16, max(ly, ry) + 20)
    lx2 = min(lx, rx);  rx2 = max(lx, rx)

    if rx2 - lx2 < 12:
        return frame

    h = 8
    cv2.line(frame, (lx2, y),   (rx2, y),   C_BRACKET, 3)
    cv2.line(frame, (lx2, y-h), (lx2, y+h), C_BRACKET, 3)
    cv2.line(frame, (rx2, y-h), (rx2, y+h), C_BRACKET, 3)
    cv2.arrowedLine(frame, (lx2+18, y), (lx2, y), C_BRACKET, 2, tipLength=0.35)
    cv2.arrowedLine(frame, (rx2-18, y), (rx2, y), C_BRACKET, 2, tipLength=0.35)

    fw = metrics.get('foot_width')
    txt = _ko(f'발너비 {fw:.2f}' if fw is not None else '발너비',
              f'Foot width {fw:.2f}' if fw is not None else 'Foot width')
    frame = draw_text_box_bgr(frame, txt, (max(5, (lx2+rx2)//2 - 70), max(16, y - 50)),
                              font_size=16, fg=(255,230,80), bg=(10,10,20), alpha=0.65)
    return frame


# ── 몸통 기울기 부족/과도 판정 ───────────────────────────────────
def evaluate_trunk_lean_issue(user_metrics, expert_metrics=None, tol_deg=None):
    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)
    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None

    if u is None or not np.isfinite(u):
        return None

    if e is None or not np.isfinite(e):
        lo, hi = 5.0, 35.0
        if u < lo:
            return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 부족', 'Trunk lean too small'), 'landmarks': [11,12,23,24], 'delta': u - lo}
        if u > hi:
            return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 과도', 'Trunk lean too large'), 'landmarks': [11,12,23,24], 'delta': u - hi}
        return None

    diff = float(u) - float(e)
    if diff > tol:
        return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 과도', 'Trunk lean too large'), 'landmarks': [11,12,23,24], 'delta': diff}
    if diff < -tol:
        return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 부족', 'Trunk lean too small'), 'landmarks': [11,12,23,24], 'delta': diff}
    return None


def trunk_status_text(user_metrics, expert_metrics=None, tol_deg=None):
    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None
    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)

    if u is None or not np.isfinite(u):
        return _ko('몸통: 측정불가', 'Trunk: N/A'), (150,150,150), None, e

    issue = evaluate_trunk_lean_issue(user_metrics, expert_metrics, tol)
    if issue is None:
        return _ko('몸통: 정상 범위', 'Trunk: OK'), (80,220,100), u, e
    return issue['message'], (80,80,255), u, e


def _point_from_vertical_angle(origin_px, length_px, angle_deg, sign):
    theta = math.radians(float(angle_deg))
    dx = sign * math.sin(theta) * length_px
    dy = -math.cos(theta) * length_px
    return (int(origin_px[0] + dx), int(origin_px[1] + dy))


# ── 몸통 기울기 zone 시각화 ──────────────────────────────────────
def draw_trunk_lean_zone(frame, lms, user_metrics, expert_metrics=None, bad_indices=None, tol_deg=None):
    """
    사람 몸 위에는 숫자 텍스트를 쓰지 않는다.
    - 초록/빨강 zone + 현재 몸통선만 표시
    - 자세한 숫자 설명은 오른쪽 HUD에서 크게 표시
    """
    if bad_indices is None:
        bad_indices = set()

    H, W = frame.shape[:2]
    sc_n = midp(lms, 11, 12)
    hc_n = midp(lms, 23, 24)
    sc   = (int(sc_n[0]*W), int(sc_n[1]*H))
    hc   = (int(hc_n[0]*W), int(hc_n[1]*H))

    torso_px = max(int(math.sqrt((sc[0]-hc[0])**2 + (sc[1]-hc[1])**2)), 40)
    guide_len = int(torso_px * 1.35)

    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None
    if u is None or not np.isfinite(u):
        return frame
    if e is None or not np.isfinite(e):
        e = u

    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)
    lo  = max(0.0, float(e) - tol)
    hi  = min(85.0, float(e) + tol)

    issue = evaluate_trunk_lean_issue(user_metrics, expert_metrics, tol)
    is_bad = issue is not None
    color = C_TRUNK_ERR if is_bad else C_TRUNK_OK

    sign = 1 if (sc[0] - hc[0]) >= 0 else -1

    p_lo  = _point_from_vertical_angle(hc, guide_len, lo, sign)
    p_hi  = _point_from_vertical_angle(hc, guide_len, hi, sign)
    p_mid = _point_from_vertical_angle(hc, guide_len, float(e), sign)

    overlay = frame.copy()
    poly = np.array([hc, p_lo, p_hi], dtype=np.int32)
    cv2.fillPoly(overlay, [poly], color)
    frame = cv2.addWeighted(overlay, 0.14, frame, 0.86, 0)

    cv2.line(frame, hc, p_lo, color, 2, cv2.LINE_AA)
    cv2.line(frame, hc, p_hi, color, 2, cv2.LINE_AA)
    cv2.line(frame, hc, p_mid, C_TRUNK_REF, 1, cv2.LINE_AA)

    # 현재 사용자 몸통 중심선
    cv2.line(frame, hc, sc, color, 5, cv2.LINE_AA)
    cv2.circle(frame, hc, 6, color, -1, cv2.LINE_AA)
    cv2.circle(frame, sc, 6, color, -1, cv2.LINE_AA)
    return frame


# 이전 함수명 호환용 alias
def draw_trunk_lean_line(frame, lms, metrics, bad_indices=None):
    return draw_trunk_lean_zone(frame, lms, metrics, None, bad_indices)


# ── threshold 초과 관절에 빨간 원 ────────────────────────────────
def draw_issue_markers(frame, lms, deltas, max_items=3):
    """
    현재 view에서 실제 오류판정에 쓰는 지표만 표시한다.
    측면영상에서 foot_width 같은 정면용 지표가 빨간 원을 만드는 문제를 방지.
    """
    H, W = frame.shape[:2]
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    items = sorted(
        [
            (abs(d) / max(rt.DELTA_THRESHOLDS.get(k, 999), 1e-6), k, d)
            for k, d in deltas.items()
            if k in active
        ],
        reverse=True
    )[:max_items]

    for ratio, key, delta in items:
        if ratio < 1.0:
            continue
        for idx in rt.DELTA_TO_LM.get(key, []):
            p = lms[idx]
            v = float(getattr(p, 'visibility', 1.0))
            if v < VIS_THR:
                continue
            cx = int(p.x * W)
            cy = int(p.y * H)
            cv2.circle(frame, (cx, cy), 13, C_ERR_CIR, 2, cv2.LINE_AA)
            cv2.circle(frame, (cx, cy),  4, C_ERR_CIR, -1, cv2.LINE_AA)
    return frame


# ── 상단 피드백 배너 ─────────────────────────────────────────────
def draw_feedback_banner(frame, feedback_text, issue_key=None,
                         user_metrics=None, expert_metrics=None):
    if not feedback_text:
        feedback_text = _ko('정상 범위', 'OK')

    is_ok = any(w in str(feedback_text) for w in ['정상', 'OK', '좋', 'safe'])
    fg = (120, 245, 150) if is_ok else (255, 130, 130)

    # 몸통 상태는 HUD에서 충분히 설명하므로 배너는 한 줄만
    return draw_text_box_bgr(
        frame,
        str(feedback_text),
        xy=(16, 52),
        font_size=20,
        fg=fg,
        bg=(18, 18, 28),
        alpha=0.74
    )


# ── PIL 기반 HUD 패널 ────────────────────────────────────────────
HUD_METRICS_SIDE = [
    # (한글명, 영어명, metric key, unit)
    ('무릎각',        'Knee angle',      'knee_angle',     'deg'),
    ('엉덩이각',      'Hip angle',       'hip_angle',      'deg'),
    ('몸통기울기',    'Trunk lean',      'trunk_lean',     'deg'),
    ('머리-골반정렬', 'Head-hip line',   'head_hip_line',  ''),
    ('발바닥밀착',    'Foot flatness',   'foot_flatness',  ''),
]

HUD_METRICS_FRONT45 = HUD_METRICS_SIDE + [
    ('발너비',        'Foot width',      'foot_width',     ''),
    ('양손높이차',    'Hand height diff','hand_height_diff',''),
]

def _fmt_val(v, unit):
    if v is None or not isinstance(v, (int, float, np.floating)) or not np.isfinite(v):
        return 'N/A'
    return f'{float(v):.1f}°' if unit == 'deg' else f'{float(v):.2f}'


def make_hud_panel(panel_w, panel_h, user_metrics, deltas, count,
                   feedback_text, issue_key, fps, ex_idx, ex_total,
                   exercise='squat'):
    img  = PilImage.new('RGB', (panel_w, panel_h), (18, 18, 28))
    drw  = ImageDraw.Draw(img)

    fTITLE = _fnt(28)
    fBIG   = _fnt(24)
    fMD    = _fnt(17)
    fSM    = _fnt(14)

    y = 14
    EX = {'squat':'SQUAT','deadlift':'DEADLIFT','bench':'BENCH PRESS'}

    # 제목
    drw.text((14, y), EX.get(exercise, exercise.upper()), font=fTITLE, fill=(240, 205, 80))
    y += 42

    # 횟수
    drw.text((14, y), _ko('현재 횟수', 'Reps'), font=fMD, fill=(180,180,190))
    drw.text((panel_w-70, y-8), str(count), font=fTITLE, fill=(80, 220, 255))
    y += 42

    # 몸통 상태 요약
    trunk_msg, trunk_col_bgr, u_tr, e_tr = trunk_status_text(user_metrics, None, TRUNK_LEAN_TOL_DEG)
    # expert값은 deltas에서 복원
    if u_tr is None:
        u_tr = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    if u_tr is not None and 'trunk_lean' in deltas:
        e_tr = u_tr - deltas.get('trunk_lean', 0)

    trunk_col_rgb = (trunk_col_bgr[2], trunk_col_bgr[1], trunk_col_bgr[0])
    drw.rounded_rectangle([10, y, panel_w-10, y+68], radius=12,
                          fill=(34,34,48), outline=trunk_col_rgb, width=2)
    drw.text((22, y+8), trunk_msg, font=fBIG, fill=trunk_col_rgb)

    detail = f"U {_fmt_val(u_tr, 'deg')}  |  E {_fmt_val(e_tr, 'deg')}  |  ±{TRUNK_LEAN_TOL_DEG:.0f}°"
    drw.text((22, y+42), detail, font=fSM, fill=(205,205,215))
    y += 82

    # 진행 정보
    prog = _ko(f'전문가 프레임 {ex_idx+1}/{ex_total}   FPS {fps:.1f}',
               f'Expert {ex_idx+1}/{ex_total}   FPS {fps:.1f}')
    drw.text((14, y), prog, font=fSM, fill=(120,120,130))
    y += 25
    drw.line([(10,y),(panel_w-10,y)], fill=(60,60,80), width=1)
    y += 14

    # 지표 목록: 측면에서는 측면 신뢰 지표만 표시
    vm = str(globals().get('VIDEO_VIEW_MODE', 'side')).lower()
    metric_list = HUD_METRICS_FRONT45 if vm not in ['side', '측면'] else HUD_METRICS_SIDE
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    for kr_label, en_label, key, unit in metric_list:
        if key not in active and key not in ['knee_angle', 'hip_angle', 'trunk_lean']:
            continue
        if y > panel_h - 125:
            break

        label = _ko(kr_label, en_label)
        val = user_metrics.get(key) if isinstance(user_metrics, dict) else None
        delta = deltas.get(key) if isinstance(deltas, dict) else None
        thr = rt.DELTA_THRESHOLDS.get(key)

        if val is not None and delta is not None:
            ex_val = val - delta
        else:
            ex_val = None

        bad = False
        if delta is not None and thr is not None:
            bad = abs(float(delta)) > float(thr)

        col = (255,130,130) if bad else (120,245,150)
        drw.text((14, y), label, font=fMD, fill=(225,225,230))
        drw.text((panel_w-100, y), _ko('주의', 'WARN') if bad else 'OK',
                 font=fMD, fill=col)

        y += 22
        val_line = f"U {_fmt_val(val, unit)}  /  E {_fmt_val(ex_val, unit)}"
        if delta is not None:
            val_line += f"  Δ {_fmt_val(delta, unit)}"
        drw.text((22, y), val_line, font=fSM, fill=(170,170,180))

        # delta 기준 bar
        y += 20
        if delta is not None and thr is not None and float(thr) > 0:
            bw = panel_w - 44
            ratio = min(abs(float(delta)) / float(thr), 1.5) / 1.5
            drw.rectangle([22, y, 22+bw, y+6], fill=(48,48,60))
            drw.rectangle([22, y, 22+int(bw*ratio), y+6], fill=col)
        y += 22

    # 하단 피드백
    drw.line([(10, panel_h-90),(panel_w-10, panel_h-90)], fill=(60,60,80), width=1)
    fb = feedback_text or _ko('정상 범위', 'OK')
    fb_col = (120,245,150) if any(w in str(fb) for w in ['정상','OK','좋','safe']) else (255,130,130)
    drw.text((14, panel_h-78), _ko('피드백', 'Feedback'), font=fMD, fill=(180,180,190))

    max_ch = max(1, (panel_w - 28) // (15 if FONT_KR_OK else 9))
    lines = textwrap.wrap(str(fb), width=max_ch)[:2]
    if not lines:
        lines = [str(fb)[:max_ch]]
    for i, line in enumerate(lines):
        drw.text((14, panel_h-52 + i*22), line, font=fMD, fill=fb_col)

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


print('시각화 헬퍼 v4 로드 완료')
print(f'  VIDEO_VIEW_MODE = {globals().get("VIDEO_VIEW_MODE", "side")}')
print(f'  한글 폰트 = {KOREAN_FONT_PATH if FONT_KR_OK else "없음 → 영어 fallback"}')
print(f'  발너비 bracket 표시 모드 = {sorted(FOOT_BRACKET_VIEW_MODES)}')
print(f'  몸통기울기 허용오차 = ±{TRUNK_LEAN_TOL_DEG}도')


In [ ]:
# ── Cell 7.5: v5 업그레이드 오버라이드 ─────────────────────────
# 목표:
# 1) '값이 큽니다/작습니다' 대신 사람이 이해하기 쉬운 피드백 문장 사용
# 2) neck_lean처럼 측면영상에서 흔들리는 지표는 핵심 오류판정에서 제외
# 3) 피드백이 프레임마다 깜빡이지 않도록 안정화(stabilizer) 적용
# 4) HUD에 U/E/Δ 의미와 현재 구간(phase)을 표시

import textwrap
from collections import deque

# ── 측면영상에서 너무 불안정한 지표 제외 ───────────────────────
# neck_lean은 얼굴/머리 landmark 흔들림 영향이 커서 발표 영상에서 "목머리각 값이..." 같은
# 어색한 피드백을 만들 가능성이 높다. 필요하면 아래 remove 목록에서 빼면 된다.
if str(globals().get('VIDEO_VIEW_MODE', 'side')).lower() in ['side', '측면']:
    _drop_keys = {'neck_lean', 'foot_width', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'}
    rt.EXERCISE_METRICS[EXERCISE] = [
        k for k in rt.EXERCISE_METRICS.get(EXERCISE, []) if k not in _drop_keys
    ]

# 발표용 핵심 오류 후보. 너무 많은 지표가 동시에 뜨지 않도록 1순위 후보를 제한한다.
V5_PRIORITY_KEYS = ['trunk_lean', 'knee_angle', 'hip_angle', 'foot_flatness', 'head_hip_line']

V5_LABELS = {
    'trunk_lean':     _ko('몸통 기울기', 'Trunk lean'),
    'knee_angle':     _ko('무릎각', 'Knee angle'),
    'hip_angle':      _ko('엉덩이각', 'Hip angle'),
    'foot_flatness':  _ko('발바닥 밀착', 'Foot contact'),
    'head_hip_line':  _ko('머리-골반 정렬', 'Head-hip line'),
}


def _safe_float(x, default=None):
    try:
        x = float(x)
        return x if np.isfinite(x) else default
    except Exception:
        return default


def phase_from_metrics(metrics):
    """무릎각 기반의 간단한 스쿼트 구간 표시."""
    k = _safe_float(metrics.get('knee_angle') if isinstance(metrics, dict) else None)
    if k is None:
        return _ko('구간 인식 중', 'Detecting phase')
    if k >= 150:
        return _ko('준비/상승완료', 'Ready / top')
    if k <= 85:
        return _ko('최저점', 'Bottom')
    if k <= 120:
        return _ko('하강·상승 중간', 'Mid phase')
    return _ko('하강/상승 전환', 'Transition')


def explain_delta_feedback(key, delta, user_metrics=None, expert_metrics=None):
    """기존 '값이 큼/작음' 대신 자세 의미 중심 피드백."""
    d = _safe_float(delta, 0.0)
    ad = abs(d)

    if key == 'trunk_lean':
        return _ko('몸통 기울기 과도' if d > 0 else '몸통 기울기 부족',
                   'Trunk lean too large' if d > 0 else 'Trunk lean too small')
    if key == 'knee_angle':
        # knee_angle이 크면 무릎이 덜 접힌 것, 작으면 더 많이 접힌 것에 가깝다.
        return _ko('무릎이 전문가보다 덜 굽혀짐' if d > 0 else '무릎이 전문가보다 많이 굽혀짐',
                   'Knee bends less than expert' if d > 0 else 'Knee bends more than expert')
    if key == 'hip_angle':
        return _ko('엉덩이 접힘이 전문가와 다름', 'Hip angle differs from expert')
    if key == 'foot_flatness':
        return _ko('발바닥/뒤꿈치 접지 확인', 'Check foot contact')
    if key == 'head_hip_line':
        return _ko('머리-골반 정렬 확인', 'Check head-hip alignment')

    label = V5_LABELS.get(key, rt.DISPLAY_LABELS.get(key, key))
    return _ko(f'{label} 차이 확인', f'Check {label}')

# rt.delta_feedback을 항상 v5 문장으로 덮어쓴다.
rt.delta_feedback = explain_delta_feedback


def choose_v5_issue(metrics_now, ex_metrics, deltas, rule_issue=None):
    """
    현재 프레임에서 가장 설명하기 좋은 오류 하나만 선택한다.
    - 몸통기울기는 별도 zone과 연결되므로 최우선
    - neck_lean 등 불안정 지표는 제외
    - threshold를 넘은 delta만 후보로 사용
    """
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    # 1) 몸통 기울기: 시각화 zone과 직접 연결되어 가장 직관적
    if 'trunk_lean' in active:
        trunk_issue = evaluate_trunk_lean_issue(metrics_now, ex_metrics, TRUNK_LEAN_TOL_DEG)
        if trunk_issue:
            trunk_issue['priority'] = 100
            trunk_issue['ratio'] = abs(_safe_float(trunk_issue.get('delta'), 0.0)) / max(TRUNK_LEAN_TOL_DEG, 1e-6)
            return trunk_issue

    # 2) delta 기반 핵심 후보
    candidates = []
    for key in V5_PRIORITY_KEYS:
        if key not in active:
            continue
        if key not in deltas:
            continue
        thr = _safe_float(rt.DELTA_THRESHOLDS.get(key))
        d = _safe_float(deltas.get(key))
        if thr is None or thr <= 0 or d is None:
            continue
        ratio = abs(d) / max(thr, 1e-6)
        if ratio < 1.0:
            continue

        # standing/top 구간에서는 head/foot 지표가 튀기 쉬워서 우선순위를 낮춘다.
        k = _safe_float(metrics_now.get('knee_angle'))
        top_phase = k is not None and k >= 150
        base_priority = {
            'knee_angle': 80,
            'hip_angle': 70,
            'foot_flatness': 58 if not top_phase else 35,
            'head_hip_line': 45 if not top_phase else 25,
            'trunk_lean': 100,
        }.get(key, 40)

        candidates.append({
            'key': key,
            'message': explain_delta_feedback(key, d, metrics_now, ex_metrics),
            'landmarks': rt.DELTA_TO_LM.get(key, []),
            'delta': d,
            'ratio': ratio,
            'priority': base_priority + min(ratio, 2.0) * 5,
        })

    if candidates:
        return sorted(candidates, key=lambda x: x.get('priority', 0), reverse=True)[0]

    # 3) 기존 rule 기반 issue가 있으면 fallback
    if rule_issue:
        out = dict(rule_issue)
        out.setdefault('priority', 30)
        out.setdefault('ratio', 1.0)
        return out

    return None


class FeedbackStabilizer:
    """같은 오류가 몇 프레임 연속 나올 때만 피드백을 바꿔 깜빡임을 줄인다."""
    def __init__(self, min_frames=3, hold_frames=8):
        self.min_frames = int(min_frames)
        self.hold_frames = int(hold_frames)
        self.candidate_key = None
        self.candidate_msg = None
        self.candidate_count = 0
        self.stable_issue = None
        self.hold_left = 0

    def update(self, issue):
        if issue is None:
            self.candidate_key = None
            self.candidate_msg = None
            self.candidate_count = 0
            if self.stable_issue is not None and self.hold_left > 0:
                self.hold_left -= 1
                return self.stable_issue
            self.stable_issue = None
            return None

        key = issue.get('key')
        msg = issue.get('message')
        same = (key == self.candidate_key and msg == self.candidate_msg)
        if same:
            self.candidate_count += 1
        else:
            self.candidate_key = key
            self.candidate_msg = msg
            self.candidate_count = 1

        if self.stable_issue is None or self.candidate_count >= self.min_frames:
            self.stable_issue = issue
            self.hold_left = self.hold_frames
        return self.stable_issue


def draw_feedback_banner(frame, feedback_text, issue_key=None,
                         user_metrics=None, expert_metrics=None, phase_text=None):
    """상단 배너를 한 줄 요약 + 현재 구간으로 표시."""
    fb = feedback_text or _ko('정상 범위', 'OK')
    phase = phase_text or phase_from_metrics(user_metrics or {})
    is_ok = any(w in str(fb) for w in ['정상', 'OK', '좋', 'safe'])
    fg = (120, 245, 150) if is_ok else (255, 130, 130)
    line1 = f'{phase}  |  {fb}'
    return draw_text_box_bgr(
        frame, line1, xy=(16, 52), font_size=21,
        fg=fg, bg=(18, 18, 28), alpha=0.78
    )


def draw_issue_markers(frame, lms, deltas, max_items=2):
    """핵심 오류 후보 관절만 표시. 빨간 원이 너무 많이 생기는 것을 방지."""
    H, W = frame.shape[:2]
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
    allowed = [k for k in V5_PRIORITY_KEYS if k in active]

    items = []
    for k in allowed:
        if k not in deltas:
            continue
        thr = _safe_float(rt.DELTA_THRESHOLDS.get(k))
        d = _safe_float(deltas.get(k))
        if thr is None or thr <= 0 or d is None:
            continue
        ratio = abs(d) / max(thr, 1e-6)
        if ratio >= 1.0:
            items.append((ratio, k, d))
    items = sorted(items, reverse=True)[:max_items]

    for ratio, key, delta in items:
        for idx in rt.DELTA_TO_LM.get(key, []):
            p = lms[idx]
            v = float(getattr(p, 'visibility', 1.0))
            if v < VIS_THR:
                continue
            cx = int(p.x * W)
            cy = int(p.y * H)
            cv2.circle(frame, (cx, cy), 14, C_ERR_CIR, 2, cv2.LINE_AA)
            cv2.circle(frame, (cx, cy),  4, C_ERR_CIR, -1, cv2.LINE_AA)
    return frame


def make_hud_panel(panel_w, panel_h, user_metrics, deltas, count,
                   feedback_text, issue_key, fps, ex_idx, ex_total,
                   exercise='squat', expert_metrics=None, phase_text=None):
    """v5 HUD: U/E/Δ 의미, 구간, 핵심 지표만 크게 표시."""
    img  = PilImage.new('RGB', (panel_w, panel_h), (18, 18, 28))
    drw  = ImageDraw.Draw(img)

    fTITLE = _fnt(28)
    fBIG   = _fnt(23)
    fMD    = _fnt(16)
    fSM    = _fnt(13)

    y = 14
    EX = {'squat':'SQUAT','deadlift':'DEADLIFT','bench':'BENCH PRESS'}

    # 제목/횟수
    drw.text((14, y), EX.get(exercise, exercise.upper()), font=fTITLE, fill=(240, 205, 80))
    drw.text((panel_w-72, y+2), str(count), font=fTITLE, fill=(80, 220, 255))
    drw.text((panel_w-118, y+12), _ko('횟수', 'rep'), font=fSM, fill=(165,165,175))
    y += 44

    # 구간 + 용어 설명
    phase = phase_text or phase_from_metrics(user_metrics or {})
    drw.rounded_rectangle([10, y, panel_w-10, y+38], radius=10, fill=(31,31,45), outline=(65,65,85), width=1)
    drw.text((22, y+9), phase, font=fMD, fill=(230,230,235))
    y += 48
    drw.text((14, y), _ko('U=사용자  E=전문가  Δ=차이', 'U=user  E=expert  Δ=difference'),
             font=fSM, fill=(145,145,155))
    y += 25

    # 핵심 피드백 카드
    fb = feedback_text or _ko('정상 범위', 'OK')
    ok = any(w in str(fb) for w in ['정상', 'OK', '좋', 'safe'])
    card_col = (95, 220, 120) if ok else (255, 115, 115)
    drw.rounded_rectangle([10, y, panel_w-10, y+70], radius=13,
                          fill=(34,34,50), outline=card_col, width=2)
    drw.text((22, y+9), _ko('현재 핵심 피드백', 'Main feedback'), font=fSM, fill=(180,180,190))
    # 너무 긴 문장 줄바꿈
    lines = textwrap.wrap(str(fb), width=max(8, (panel_w-44)//15))[:2]
    if not lines:
        lines = [str(fb)]
    for i, line in enumerate(lines):
        drw.text((22, y+31+i*21), line, font=fMD, fill=card_col)
    y += 84

    # 진행 정보
    prog = _ko(f'전문가 프레임 {ex_idx+1}/{ex_total}   FPS {fps:.1f}',
               f'Expert {ex_idx+1}/{ex_total}   FPS {fps:.1f}')
    drw.text((14, y), prog, font=fSM, fill=(115,115,125))
    y += 24
    drw.line([(10,y),(panel_w-10,y)], fill=(60,60,80), width=1)
    y += 14

    # 지표 목록: 발표용으로 5개만
    metric_list = [
        ('무릎각',        'Knee angle',      'knee_angle',     'deg'),
        ('엉덩이각',      'Hip angle',       'hip_angle',      'deg'),
        ('몸통기울기',    'Trunk lean',      'trunk_lean',     'deg'),
        ('머리-골반정렬', 'Head-hip line',   'head_hip_line',  ''),
        ('발바닥밀착',    'Foot contact',    'foot_flatness',  ''),
    ]

    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
    for kr_label, en_label, key, unit in metric_list:
        if key not in active and key not in ['knee_angle', 'hip_angle', 'trunk_lean']:
            continue
        if y > panel_h - 80:
            break

        label = _ko(kr_label, en_label)
        val = user_metrics.get(key) if isinstance(user_metrics, dict) else None
        ex_val = None
        if isinstance(expert_metrics, dict):
            ex_val = expert_metrics.get(key)
        if ex_val is None and isinstance(deltas, dict) and key in deltas and val is not None:
            ex_val = val - deltas.get(key, 0)

        delta = None
        if val is not None and ex_val is not None:
            try:
                delta = float(val) - float(ex_val)
            except Exception:
                delta = None
        elif isinstance(deltas, dict):
            delta = deltas.get(key)

        thr = rt.DELTA_THRESHOLDS.get(key)
        bad = False
        if delta is not None and thr is not None:
            bad = abs(float(delta)) > float(thr)
        col = (255,130,130) if bad else (120,245,150)

        drw.text((14, y), label, font=fMD, fill=(225,225,230))
        drw.text((panel_w-74, y), _ko('주의', 'WARN') if bad else 'OK', font=fMD, fill=col)
        y += 21

        val_line = f"U {_fmt_val(val, unit)} / E {_fmt_val(ex_val, unit)}"
        if delta is not None:
            val_line += f" / Δ {_fmt_val(delta, unit)}"
        drw.text((22, y), val_line, font=fSM, fill=(172,172,182))
        y += 19

        if delta is not None and thr is not None and float(thr) > 0:
            bw = panel_w - 44
            ratio = min(abs(float(delta)) / float(thr), 1.5) / 1.5
            drw.rectangle([22, y, 22+bw, y+5], fill=(48,48,60))
            drw.rectangle([22, y, 22+int(bw*ratio), y+5], fill=col)
        y += 18

    # 하단 설명
    drw.line([(10, panel_h-64),(panel_w-10, panel_h-64)], fill=(60,60,80), width=1)
    note = _ko('빨간 원 = 현재 우선 확인 관절', 'Red circle = joint to check first')
    drw.text((14, panel_h-48), note, font=fSM, fill=(165,165,175))
    note2 = _ko('측면영상: 발너비/양손거리 판정 제외', 'Side view: width metrics excluded')
    drw.text((14, panel_h-28), note2, font=fSM, fill=(115,115,125))

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

print('v5 피드백 업그레이드 로드 완료')
print(f'  핵심 오류 후보: {V5_PRIORITY_KEYS}')
print(f'  현재 활성 지표: {rt.EXERCISE_METRICS.get(EXERCISE)}')


In [ ]:
# ── Cell 8: 모델 / 전문가 JSON / 영상 로드 & 레이아웃 계산 ────────
# 핵심 수정:
# - 원본 영상을 HALF_W로 찌그러뜨리지 않음.
# - USER 영역은 원본 W×H 그대로 유지.
# - EXPERT skeleton과 HUD는 오른쪽에 별도 영역으로 추가.
# - 따라서 출력 영상 폭은 넓어지지만 원본 비율은 자연스럽게 유지됨.

rt.ensure_model(MODEL_PATH)
expert = rt.load_expert(EXERCISE, EXPERT_JSON_PATH)

ex_total       = expert['total_frames']
ex_frames_data = expert['frames']
print(f'[전문가] {ex_total}프레임, {expert.get("fps",24):.1f}fps')

print('[PoseLandmarker] 로딩 중...')
landmarker = rt.build_landmarker(MODEL_PATH)
print('[PoseLandmarker] 완료')

# probe용으로만 영상 정보 읽기
cap_probe = cv2.VideoCapture(USER_VIDEO_PATH)
if not cap_probe.isOpened():
    raise RuntimeError(f'영상 열기 실패: {USER_VIDEO_PATH}')

fps = cap_probe.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

W = int(cap_probe.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap_probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_probe.release()

print(f'[영상] {W}x{H}, {fps:.2f}fps')

# ── 출력 레이아웃 ─────────────────────────────────────────────────
# 기존 문제:
#   HALF_W = (W-PW)//2 로 사용자 영상을 강제 축소 → 화면이 세로로 찌그러짐.
# 수정:
#   USER_W = W, USER_H = H 그대로 사용.
#   오른쪽에 EXPERT_W와 HUD_W를 추가.

USER_W = W
USER_H = H

# 전문가 skeleton 영역 폭. 너무 작으면 skeleton이 답답하고, 너무 크면 출력이 과도하게 넓어짐.
EXPERT_W = min(520, max(420, int(W * 0.40)))
PW       = 410   # v5 HUD 카드/설명 공간 확보

OUT_W = USER_W + EXPERT_W + PW
OUT_H = H

print(f'[레이아웃] USER:{USER_W}x{USER_H} | EXPERT:{EXPERT_W}x{OUT_H} | HUD:{PW}x{OUT_H}')
print(f'[출력 크기] {OUT_W}x{OUT_H}')
print(f'[출력 경로] {OUTPUT_PATH}')


In [ ]:
# ── Cell 9: 상태 객체 & 렌더러 설정 ─────────────────────────────
# 핵심 수정:
# - 이전 overlay에서 쓰던 arc / metricbar / acl / ktg 등을 다시 켬.
# - rt 모듈에 없는 객체는 notebook이 멈추지 않도록 fallback 생성.

import numpy as np

if not hasattr(rt, 'WINDOW_SIZE'):
    rt.WINDOW_SIZE = 5

if not hasattr(rt, 'PoseLandmarksConns'):
    rt.PoseLandmarksConns = SKEL_CONNS


# MetricsBuffer 없을 경우 fallback
if not hasattr(rt, 'MetricsBuffer'):
    class MetricsBuffer:
        def __init__(self, window=5):
            self.window = int(window)
            self.buf = []

        def push(self, metrics):
            self.buf.append(dict(metrics))
            if len(self.buf) > self.window:
                self.buf.pop(0)

        def get_avg(self):
            if not self.buf:
                return {}

            keys = set()
            for m in self.buf:
                keys.update(m.keys())

            out = {}
            for k in keys:
                vals = []
                for m in self.buf:
                    v = m.get(k)
                    if isinstance(v, (int, float, np.floating)) and np.isfinite(v):
                        vals.append(float(v))
                if vals:
                    out[k] = float(np.mean(vals))
            return out

    rt.MetricsBuffer = MetricsBuffer
    print('보정: rt.MetricsBuffer 임시 생성 완료')


# RuleState 없을 경우 fallback
if not hasattr(rt, 'RuleState'):
    class RuleState:
        def __init__(self, exercise='squat'):
            self.exercise = exercise

        def evaluate(self, metrics):
            # 세부 rule이 없으면 delta 기반 비교만 사용
            return []

    rt.RuleState = RuleState
    print('보정: rt.RuleState 임시 생성 완료')


# SideViewRepCounter 없을 경우 fallback
if not hasattr(rt, 'SideViewRepCounter'):
    class SideViewRepCounter:
        def __init__(self, exercise='squat'):
            self.exercise = exercise
            self.count = 0
            self.was_down = False

        def update(self, metrics):
            if self.exercise == 'squat':
                angle = metrics.get('knee_angle')
                if angle is None:
                    return False

                # squat 기준 간단 카운터
                # 내려감: 무릎각 작아짐 / 올라옴: 무릎각 커짐
                low_thr  = 105
                high_thr = 150

                if angle < low_thr:
                    self.was_down = True

                if self.was_down and angle > high_thr:
                    self.count += 1
                    self.was_down = False
                    return True

            return False

    rt.SideViewRepCounter = SideViewRepCounter
    print('보정: rt.SideViewRepCounter 임시 생성 완료')


if not hasattr(rt, 'pick_worst_issue'):
    def pick_worst_issue(issues):
        if not issues:
            return None
        return issues[0]

    rt.pick_worst_issue = pick_worst_issue
    print('보정: rt.pick_worst_issue 임시 생성 완료')


if not hasattr(rt, 'compute_deltas'):
    def compute_deltas(user_metrics, expert_metrics, exercise='squat'):
        deltas = {}
        keys = rt.EXERCISE_METRICS.get(exercise, list(user_metrics.keys()))

        for k in keys:
            if k not in user_metrics or k not in expert_metrics:
                continue

            uv = user_metrics.get(k)
            ev = expert_metrics.get(k)

            if uv is None or ev is None:
                continue

            try:
                deltas[k] = float(uv) - float(ev)
            except Exception:
                pass

        return deltas

    rt.compute_deltas = compute_deltas
    print('보정: rt.compute_deltas 임시 생성 완료')


if not hasattr(rt, 'select_worst_delta'):
    def select_worst_delta(deltas):
        if not deltas:
            return None

        worst_key = None
        worst_delta = None
        worst_ratio = 0.0

        for k, d in deltas.items():
            thr = rt.DELTA_THRESHOLDS.get(k)
            if thr is None or thr <= 0:
                continue

            ratio = abs(float(d)) / max(float(thr), 1e-6)
            if ratio > worst_ratio:
                worst_ratio = ratio
                worst_key = k
                worst_delta = d

        if worst_key is None or worst_ratio < 1.0:
            return None

        return worst_key, worst_delta

    rt.select_worst_delta = select_worst_delta
    print('보정: rt.select_worst_delta 임시 생성 완료')


if not hasattr(rt, 'delta_feedback'):
    def delta_feedback(key, delta):
        label = rt.DISPLAY_LABELS.get(key, key)

        if delta > 0:
            return f'{label} 값이 전문가보다 큽니다'
        return f'{label} 값이 전문가보다 작습니다'

    rt.delta_feedback = delta_feedback
    print('보정: rt.delta_feedback 임시 생성 완료')


# 상태 객체 생성
buf         = rt.MetricsBuffer(window=rt.WINDOW_SIZE)
rules       = rt.RuleState(EXERCISE)
rep_counter = rt.SideViewRepCounter(EXERCISE)

renderer = fo.FeedbackRenderer(fps=fps)


# ── 기존 시각화 레이어 복구 ─────────────────────────────────────
# Claude 코드에서 phase/tempo만 켜져 있어서 무릎각 arc, metric bar, acl, ktg 등이 사라졌음.
# feedback_overlay.py 버전에 따라 없는 레이어는 자동 skip.
# 기본적으로 예전 노트북에서 쓰던 검증된 레이어만 켠다.
# bar_proxy / heatmap / trail 같은 이름은 feedback_overlay.py 버전에 따라 없을 수 있으므로
# renderer 내부 registry에서 확인될 때만 켠다.
# metricbar는 한글/숫자가 작게 겹쳐 깨져 보이므로 오른쪽 HUD로 대체.
# ktg도 버전에 따라 작은 텍스트가 겹치면 아래 리스트에서 빼도 된다.
CORE_LAYERS = ['phase', 'tempo', 'depth', 'arc', 'acl', 'ktg']
OPTIONAL_LAYERS = ['bar_proxy', 'heatmap', 'trail']

available_layers = None
for attr in ['layers', 'layer_map', 'registry', 'renderers', 'available_layers']:
    obj = getattr(renderer, attr, None)
    if isinstance(obj, dict):
        available_layers = set(obj.keys())
        break
    if isinstance(obj, (set, list, tuple)):
        available_layers = set(obj)
        break

if available_layers is None:
    VISIBLE_LAYERS = CORE_LAYERS
else:
    VISIBLE_LAYERS = [x for x in CORE_LAYERS + OPTIONAL_LAYERS if x in available_layers]

enabled_layers = []
skipped_layers = []

for layer in VISIBLE_LAYERS:
    try:
        renderer.enable(layer)
        enabled_layers.append(layer)
    except Exception as e:
        skipped_layers.append((layer, str(e)))

# depth 목표선 설정
try:
    expert_target_y_norm = fo.extract_expert_depth_target(ex_frames_data, EXERCISE)
    renderer.depth_line.set_expert_target(expert_target_y_norm, OUT_H)
except Exception as e:
    print(f'[참고] depth 목표선 설정 skip: {e}')

print('상태 객체 / 렌더러 설정 완료')
print(f'  종목: {EXERCISE} | MetricsBuffer 윈도우: {rt.WINDOW_SIZE}')
print(f'  ON 레이어: {enabled_layers}')
if skipped_layers:
    print(f'  SKIP 레이어: {[x[0] for x in skipped_layers]}')


In [ ]:
# ── Cell 10: 메인 처리 루프 v5 ───────────────────────────────────
# v5 핵심:
# - 오류 후보를 1개로 안정화해서 피드백 깜빡임 감소
# - '값이 큼/작음' 대신 의미 있는 문장으로 표시
# - HUD에 현재 구간, U/E/Δ 의미, 핵심 피드백을 크게 표시

# Cell 10만 다시 실행해도 되도록 cap/out을 여기서 새로 연다.
cap = cv2.VideoCapture(USER_VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f'영상 열기 실패: {USER_VIDEO_PATH}')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (OUT_W, OUT_H))

if not out.isOpened():
    raise RuntimeError(f'VideoWriter 열기 실패: {OUTPUT_PATH}')

frame_idx = 0
t_start   = time.perf_counter()
feedback_stabilizer = FeedbackStabilizer(min_frames=3, hold_frames=8)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if rt.MIRROR_CAMERA:
        frame = cv2.flip(frame, 1)

    # 원본 비율 유지
    user_canvas = frame.copy()

    elapsed_ms = int(frame_idx * 1000 / fps)

    # 전문가 프레임 인덱스 & 지표
    ex_idx     = rt.get_expert_frame_idx(elapsed_ms, expert)
    ex_pack    = ex_frames_data[ex_idx]
    ex_lms_raw = ex_pack['landmarks']
    ex_metrics = compute_expert_metrics_plus(
        ex_lms_raw,
        ex_pack.get('metrics', {}),
        EXERCISE
    )

    # 사용자 landmark 추출: 원본 frame 기준
    lms = rt.extract_landmarks(landmarker, user_canvas, elapsed_ms)

    feedback    = ''
    issue_key   = None
    bad_indices = set()
    deltas      = {}
    user_avg    = {}
    raw         = {}
    metrics_now = {}
    phase_text  = _ko('구간 인식 중', 'Detecting phase')

    if lms:
        try:
            raw = compute_user_metrics_plus(lms, EXERCISE)
            buf.push(raw)
            user_avg = buf.get_avg()
        except Exception as e:
            feedback = f'지표 계산 오류: {e}'
            print('[지표 계산 오류]', e)
            user_avg = {}

        metrics_now = user_avg if user_avg else raw
        phase_text  = phase_from_metrics(metrics_now)

        if metrics_now:
            # 횟수 카운트
            if rep_counter.update(metrics_now):
                print(f'[{EXERCISE.upper()}] {rep_counter.count}회')

            # 전문가 delta 계산 후 현재 view에서 실제 사용하는 지표만 유지
            deltas = rt.compute_deltas(metrics_now, ex_metrics, EXERCISE)
            active_keys = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
            deltas = {k: v for k, v in deltas.items() if k in active_keys}

            # 기존 rule 기반 issue는 fallback으로만 사용
            rule_issues = rules.evaluate(metrics_now)
            rule_issue  = rt.pick_worst_issue(rule_issues)

            # v5: 가장 설명하기 좋은 issue 1개 선택 후 안정화
            candidate_issue = choose_v5_issue(metrics_now, ex_metrics, deltas, rule_issue)
            stable_issue = feedback_stabilizer.update(candidate_issue)

            if stable_issue:
                issue_key   = stable_issue.get('key')
                feedback    = stable_issue.get('message', '')
                bad_indices = set(stable_issue.get('landmarks', []))
            else:
                feedback = _ko('정상 범위', 'OK')

        # 기존 feedback_overlay 렌더
        try:
            user_canvas = renderer.render(
                frame            = user_canvas,
                user_metrics_raw = metrics_now,
                expert_metrics   = ex_metrics,
                deltas           = deltas,
                lms              = lms,
                connections      = rt.PoseLandmarksConns,
                metric_to_lm     = rt.DELTA_TO_LM,
                exercise         = EXERCISE
            )
        except Exception as e:
            # renderer 하나가 실패해도 전체 영상 생성은 계속 진행
            print(f'[renderer 오류] frame={frame_idx}: {e}')

        # 추가 레이어 1: 발너비 bracket
        # 측면이면 VIDEO_VIEW_MODE='side'라서 자동으로 숨김.
        user_canvas = draw_foot_bracket(
            user_canvas, lms, metrics_now, view_mode=VIDEO_VIEW_MODE
        )

        # 추가 레이어 2: 몸통기울기 허용범위 zone
        user_canvas = draw_trunk_lean_zone(
            user_canvas, lms,
            metrics_now,
            ex_metrics,
            bad_indices,
            tol_deg=TRUNK_LEAN_TOL_DEG
        )

        # 추가 레이어 3: threshold 초과 핵심 관절 원 표시
        user_canvas = draw_issue_markers(
            user_canvas, lms, deltas, max_items=2
        )

        # 추가 레이어 4: 큰 피드백 배너
        user_canvas = draw_feedback_banner(
            user_canvas, feedback, issue_key, metrics_now, ex_metrics, phase_text=phase_text
        )

    else:
        feedback = _ko('자세를 인식할 수 없습니다. 전신이 보이게 해주세요.',
                       'Pose not detected. Show full body.')

    # 사용자 레이블
    cv2.putText(
        user_canvas, 'USER',
        (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX, 0.75,
        (60, 210, 255), 2, cv2.LINE_AA
    )

    # 전문가 skeleton canvas
    expert_canvas = draw_expert_on_canvas(
        EXPERT_W, OUT_H, ex_lms_raw, bad_indices
    )

    # HUD
    elapsed = time.perf_counter() - t_start
    cur_fps = (frame_idx + 1) / max(elapsed, 1e-6)

    hud_panel = make_hud_panel(
        panel_w        = PW,
        panel_h        = OUT_H,
        user_metrics   = user_avg if user_avg else raw,
        deltas         = deltas,
        count          = rep_counter.count,
        feedback_text  = feedback,
        issue_key      = issue_key,
        fps            = cur_fps,
        ex_idx         = ex_idx,
        ex_total       = ex_total,
        exercise       = EXERCISE,
        expert_metrics = ex_metrics,
        phase_text     = phase_text
    )

    # 최종 합성: [원본 USER | EXPERT | HUD]
    final_frame = np.hstack([user_canvas, expert_canvas, hud_panel])

    # 영역 구분선
    cv2.line(final_frame, (USER_W, 0), (USER_W, OUT_H), (50, 50, 70), 2)
    cv2.line(final_frame, (USER_W + EXPERT_W, 0), (USER_W + EXPERT_W, OUT_H), (50, 50, 70), 2)

    out.write(final_frame)
    frame_idx += 1

    if frame_idx % 30 == 0:
        print(f'[처리 중] {frame_idx}f | rep={rep_counter.count} | feedback={feedback} | fps={cur_fps:.1f}')

cap.release()
out.release()

try:
    landmarker.close()
except Exception:
    pass

print(f'\n완료! 총 {frame_idx} frames')
print(f'총 횟수: {rep_counter.count}회')
print(f'출력: {OUTPUT_PATH}')


In [ ]:
# ── Cell 11: 저장 확인만 하기 ─────────────────────────
import os

if not os.path.exists(OUTPUT_PATH):
    print('출력 파일 없음 -- Cell 10 정상 실행 여부 확인')
else:
    size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
    print(f'저장 완료!')
    print(f'파일 크기: {size_mb:.1f} MB')
    print(f'저장 위치: {OUTPUT_PATH}')

In [ ]:
# ── Cell 12: 단일 프레임 디버그 (선택 실행) ──────────────────────
# 메인 루프 실행 전에 시각화를 점검할 때 사용.
# Cell 8 (모델/영상 로드) 이후에 실행 가능.

from matplotlib import pyplot as plt


def debug_frame(video_path, frame_no=60, ex_fi=30):
    """
    지정 프레임의 사용자/전문가 시각화를 matplotlib으로 확인.
    발너비 bracket, 몸통기울기 zone, 전문가 skeleton이 제대로 나오는지 점검용.
    """
    cap_d = cv2.VideoCapture(video_path)
    cap_d.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ret, frame = cap_d.read()
    cap_d.release()
    if not ret:
        print(f'프레임 {frame_no} 읽기 실패')
        return

    uc = frame.copy()
    elapsed_ms = int(frame_no * 1000 / fps)
    lms_d = rt.extract_landmarks(landmarker, uc, elapsed_ms)

    ex_fi = max(0, min(int(ex_fi), len(ex_frames_data)-1))
    ex_pack_d = ex_frames_data[ex_fi]
    ex_lms_d  = ex_pack_d['landmarks']
    ex_met_d  = compute_expert_metrics_plus(ex_lms_d, ex_pack_d.get('metrics', {}), EXERCISE)

    raw_d = {}
    if lms_d:
        raw_d = compute_user_metrics_plus(lms_d, EXERCISE)
        trunk_issue = evaluate_trunk_lean_issue(raw_d, ex_met_d, TRUNK_LEAN_TOL_DEG)
        bad_d = set(trunk_issue.get('landmarks', [])) if trunk_issue else set()

        uc = draw_foot_bracket(uc, lms_d, raw_d, view_mode=VIDEO_VIEW_MODE)
        uc = draw_trunk_lean_zone(uc, lms_d, raw_d, ex_met_d, bad_d, tol_deg=TRUNK_LEAN_TOL_DEG)

        print('=== 지표 계산 결과 ===')
        print(f'VIDEO_VIEW_MODE: {VIDEO_VIEW_MODE}')
        print(f'발너비 bracket 표시 여부: {should_draw_foot_bracket(lms_d, VIDEO_VIEW_MODE)}')
        if trunk_issue:
            print(f'몸통기울기 피드백: {trunk_issue["message"]}')
        else:
            print('몸통기울기 피드백: 정상 범위')
        for k, v in sorted(raw_d.items()):
            print(f'  {k:30s}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
    else:
        print('landmark 미검출')

    ec = draw_expert_on_canvas(EXPERT_W, OUT_H, ex_lms_d, set())
    combined = np.hstack([uc, ec])

    plt.figure(figsize=(14, 7))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title(f'프레임 {frame_no} | 좌: USER  우: EXPERT')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# 60번 프레임 / 전문가 30번 프레임 확인
# debug_frame(USER_VIDEO_PATH, frame_no=60, ex_fi=30)


---
## 📌 v5 주요 수정 사항 요약

| 수정 항목 | 기존 문제 | v5 수정 방법 |
|---|---|---|
| 피드백 문장 | `값이 전문가보다 큽니다/작습니다`가 무엇을 의미하는지 직관적이지 않음 | `몸통 기울기 과도`, `무릎이 덜 굽혀짐`, `발바닥/뒤꿈치 접지 확인`처럼 의미 중심 문장으로 변환 |
| 피드백 깜빡임 | 프레임마다 최악 지표가 바뀌면서 문장이 계속 바뀜 | `FeedbackStabilizer(min_frames=3, hold_frames=8)`로 같은 오류가 몇 프레임 유지될 때만 변경 |
| 측면영상 지표 | 발너비, 양손거리, 목머리각이 측면에서 흔들리며 오검출 유발 | 측면영상에서는 `neck_lean`, `foot_width`, `hand_width`, `hand_height_diff`, `shoulder_height_diff`를 핵심판정에서 제외 |
| HUD | 숫자는 있지만 U/E/Δ 의미가 불명확함 | HUD 상단에 `U=사용자, E=전문가, Δ=차이` 설명 추가 |
| 빨간 원 | 여러 관절이 동시에 표시되어 복잡함 | 핵심 오류 후보 2개까지만 표시 |

---
## ⚙️ 조정 포인트

```python
TRUNK_LEAN_TOL_DEG = 8.0      # 몸통기울기 허용오차
FeedbackStabilizer(min_frames=3, hold_frames=8)
```

- 피드백이 너무 늦게 바뀌면 `min_frames=2`로 낮추면 됩니다.
- 피드백이 너무 자주 바뀌면 `min_frames=4`, `hold_frames=10` 정도로 높이면 됩니다.
- 측면영상에서 발너비는 실제 너비 판단이 어려우므로 계속 제외하는 것이 맞습니다.
